In [ ]:
%%bash
cd ..
# rm data/*
# rm -r logs
# rm -r src/__pycache__
tree

In [1]:
import os
import pickle
import warnings

from datetime import datetime, timezone
from pathlib import PosixPath

import hopsworks
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px

from catboost import CatBoostRegressor
from dotenv import load_dotenv
from hopsworks.project import Project
from hsfs.feature_group import FeatureGroup
from hsfs.feature_store import FeatureStore
from hsml.model_registry import ModelRegistry
from hsml.model_schema import ModelSchema, Schema
from hsml.sklearn.model import Model
from lightgbm import LGBMRegressor
from plotly.graph_objects import Figure
from xgboost import XGBRegressor

from src.config import Config
from src.feature_store_api import get_feature_group
from src.ingest import download_file
from src.train import compute_metrics, NaiveForecast, split_data, train_model
from src.transform import plot_record, tabularize_data

warnings.filterwarnings("ignore")
load_dotenv(Config.HOME_DIR / ".env")

True

In [ ]:
(
    pd.read_csv("~/Downloads/emit_ch4.csv")
    # .reset_index()
    # .sort_values(["day", "time_utc"], ascending=[False, False])
)


In [ ]:
pd.Timestamp((
    pl.read_csv("~/Downloads/emit_ch4.csv")
    .with_columns(
        (pl.col("day") + " " + pl.col("time_utc"))
        .str.to_datetime("%b %d, %Y %H:%M:%S.%f")
        .alias("datetime")
    )
    ["datetime"].max()
))

In [ ]:
# set the Pandas display options
pd.set_option("display.max_rows", 100) # max number of rows to display
pd.set_option("display.max_columns", None) # max number of columns to display

# set the Polars display options
pl.Config(tbl_rows=10) # max number of rows to display
pl.Config(tbl_cols=None) # max number of columns to display
pl.Config(fmt_str_lengths=25) # max number of characters to display for pl.Utf8 (str) dtype columns
pl.Config(fmt_table_cell_list_len=20) # max number of items to display for pl.List dtype columns

#### **`Data ingestion`**

In [ ]:
# download, validate, and pre-process raw data from 2022-01
df: pd.DataFrame = download_file(2022, 1)
df

In [ ]:
# confirm that the 'df' pd.DataFrame is free of null values and duplicates
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0

In [ ]:
# a list of select location IDs
location_ids: list[int] = [43, 90, 107]

# plot the hourly taxi rides for the selected location IDs
fig: Figure = px.line(
    df.query(f"location_id.isin({location_ids})"),
    x="pickup_datetime",
    y="rides",
    color="location_id",
    labels={
        "pickup_datetime": "Datetime", 
        "rides": "Number of taxi rides", 
        "location_id": "Location ID"
    },
    title="NYC Hourly Taxi Rides",
    template="plotly_dark"
)
fig.show()

#### **`Data transformation`**

In [ ]:
# download, validate, pre-process, and transform raw data from 2022-01 into an ML-ready dataset
# NOTE: for the 'day_of_week' feature, 0 == Monday, ..., 6 == Sunday
df: pd.DataFrame = download_file(2022, 1).pipe(tabularize_data)
df

In [ ]:
# plot a single record's (row's) lag features (blue) and target (green)
plot_record(df)

#### **`Model training and evaluation`**

In [ ]:
# download, validate, pre-process, and transform raw data from 2023-05 into an ML-ready dataset
df: pd.DataFrame = download_file(2023, 5).pipe(tabularize_data)

In [ ]:
# extract the 'best' model
model: CatBoostRegressor | LGBMRegressor | XGBRegressor = train_model(df)

# split the 'df' pd.DataFrame into train and test sets
Xtrain, ytrain, Xtest, ytest = split_data(df)

# fit the model to the train set and evaluate on the test set
if isinstance(model, CatBoostRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], early_stopping_rounds=50, verbose=False)
elif isinstance(model, LGBMRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)])
elif isinstance(model, XGBRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], verbose=False)

# output the model and its corresponding test set metrics (RMSE and R²)
display(model, compute_metrics(ytest, model.predict(Xtest)))

#### **`Feature pipeline (Hopsworks)`**

In [ ]:
# specify the starting and ending timestamps for which to ingest data
current_ts: pd.Timestamp = pd.Timestamp(datetime.now(timezone.utc)).floor("H")
end: pd.Timestamp = current_ts - pd.Timedelta(days=366)
start: pd.Timestamp = end - pd.Timedelta(days=28)

# a list of pre-processed pd.DataFrames, one per (year, month) pair
dfs: list[pd.DataFrame] = [
    download_file(year, month) 
    for year, month in zip([start.year, end.year], [start.month, end.month])
]

# (1) convert each pd.DataFrame in the 'dfs' list to a pl.DataFrame, and ...
# vertically concatenate them into a single pl.DataFrame
# (2) modify the 'pickup_datetime' column's timestamps by setting their timezone to UTC.
# Then, filter the pl.DataFrame by only keeping records (rows) whose modified timestamp is ...
# greater than or equal to the 'start' timestamp and less than or equal to the 'end' timestamp
# (3) modify the 'pickup_datetime' column's timestamps by ...
# (a) setting their timezone to UTC, ...
# (b) offsetting them forward in time by one year, and ...
# (c) converting them to integers that represent the time passed, in ms, since the Unix EPOCH
# (4) rename the 'pickup_datetime' column to 'unix_epoch_ms'
# (5) convert the pl.DataFrame to a pd.DataFrame
# NOTE: the NYC taxi data lags behind the current date by three months, so the timestamps are ...
# offset forward in time by one year to simulate a real time data ingestion scenario
# NOTE: the 'df' pd.DataFrame contains the data that will be uploaded to Hopsworks
df: pd.DataFrame = (
    pl.concat([pl.from_pandas(df) for df in dfs], how="vertical") # (1)
    .filter( # (2)
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").ge(start),
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").le(end)
    )
    .with_columns( # (3)
        pl.col("pickup_datetime")
        .dt.replace_time_zone("UTC")
        .dt.offset_by("1y")
        .dt.epoch(time_unit="ms")
    )
    .rename({"pickup_datetime": "unix_epoch_ms"}) # (4)
    .to_pandas() # (5)
)
df

In [ ]:
# NOTE: this cell transforms the 'df' pd.DataFrame to a ML-ready dataset that contains ...
# datetime features, window features (average lags), lag features, and the corresponding target
# (1) convert the 'df' pd.DataFrame to a pl.DataFrame
# (2) convert the 'unix_epoch_ms' column's entries, which are integers, to timestamps
# (3) rename the 'unix_epoch_ms' column to 'pickup_datetime'
# (4) sort the pl.DataFrame by the 'location_id' and 'pickup_datetime' columns
# (5) convert the pl.DataFrame back to a pd.DataFrame
# (6) tabularize the pd.DataFrame, that is, convert it to an ML-ready dataset containing ...
# datetime features, window features (average lags), lag features, and the corresponding target
(
    pl.from_pandas(df)
    .with_columns(pl.from_epoch(pl.col("unix_epoch_ms"), time_unit="ms"))
    .rename({"unix_epoch_ms": "pickup_datetime"})
    .sort(["location_id", "pickup_datetime"])
    .to_pandas()
    .pipe(tabularize_data)
)

In [ ]:
# connect to the Hopsworks 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project="taxi_demand_forecasting",
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# connect to the project's feature store
feature_store: FeatureStore = project.get_feature_store()

# create the feature store's feature group
# NOTE: the feature group allows the 'df' pd.DataFrame's features to be saved to the feature store
# NOTE: the 'primary_key' parameter below, which can be set equal to a single column or ...
# a list of columns from the 'df' pd.DataFrame, is used as a unique identifier for each record (row)
feature_group: FeatureGroup = feature_store.get_or_create_feature_group(
    name="univariate_time_series",
    version=1,
    description="NYC taxi rides recorded at an hourly frequency",
    primary_key=["location_id", "unix_epoch_ms"],
    event_time="unix_epoch_ms",
    online_enabled=True
)

# upload the 'df' pd.DataFrame to the 'taxi_demand_forecasting' project as a new feature group
# NOTE: the feature group's name is, 'univariate_time_series', and it can be found on the ...
# Hopsworks UI, under the 'taxi_demand_forecasting' project's 'Feature Group' tab 
feature_group.insert(df, write_options={"wait_for_job": False})

#### **`Training pipeline (Hopsworks)`**

In [ ]:
# fetch the pre-processed data from Hopsworks
processed_data: pd.DataFrame = get_feature_group().read()

# convert the pre-processed data to tabular, ML-ready data
if not processed_data.empty:
    tabular_data: pd.DataFrame = (
        pl.from_pandas(processed_data)
        .with_columns(pl.from_epoch(pl.col("unix_epoch_ms"), time_unit="ms"))
        .rename({"unix_epoch_ms": "pickup_datetime"})
        .sort(["location_id", "pickup_datetime"])
        .to_pandas()
        .pipe(tabularize_data)
    )
tabular_data

In [ ]:
# split the 'tabular_data' pd.DataFrame into train and test sets
Xtrain, ytrain, Xtest, ytest = tabular_data.pipe(split_data)

# get the test set's 'baseline' metrics
baseline_metrics: dict[str, float] = compute_metrics(ytest, NaiveForecast().predict(Xtest))

# fit the model to the train set and evaluate on the test set
if isinstance(model, CatBoostRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], early_stopping_rounds=50, verbose=False)
elif isinstance(model, LGBMRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)])
elif isinstance(model, XGBRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], verbose=False)
    
# get the model's test set metrics
metrics: dict[str, float] = compute_metrics(ytest, model.predict(Xtest))

# display the model, its test set metrics, and the test set's 'baseline' metrics
display(model, f"Metrics: {metrics}", f"Baseline metrics: {baseline_metrics}")

In [ ]:
# if the model's test set predictions are better than the naive forecast, then ...
# (1) save the model locally as an artifact, that is, ~/artifacts/model.pkl, and ...
# (2) upload the saved artifact to the Hopsworks model registry
if metrics.get("rmse") < baseline_metrics.get("rmse"):
    output_dir: PosixPath = Config.ARTIFACTS_DIR
    output_dir.mkdir(parents=True, exist_ok=True)
    pickle.dump(model, open(output_dir / "model.pkl", "wb"))

# confirm that the saved model works
# NOTE: the metrics output from this cell should match the 'Metrics' dict from the above code cell
saved_model: LGBMRegressor = pickle.load(open(Config.ARTIFACTS_DIR / "model.pkl", "rb"))
compute_metrics(ytest, saved_model.predict(Xtest))

In [ ]:
%%bash
cd ..
tree artifacts

In [ ]:
# connect to the Hopsworks 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project="taxi_demand_forecasting",
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# create an object that points to the project's model registry
model_registry: ModelRegistry = project.get_model_registry()

# upload ~/artifacts/model.pkl to the project's model registry
# NOTE: the model and its metadata can be viewed on the Hopsworks UI, under the 'taxi_demand_forecasting' 
# project's 'Model Registry' tab 
(
    model_registry.sklearn
    .create_model(
        name="one_step_forecaster",
        metrics={"rmse": metrics.get("rmse")},
        metrics=metrics,
        description=model.__str__(),
        input_example=Xtrain.sample(),
        model_schema=ModelSchema(input_schema=Schema(Xtrain), output_schema=Schema(ytrain))
    )
    .save(os.path.join(Config.ARTIFACTS_DIR, "model.pkl"))
)

In [ ]:
# add the 'location_id', 'pickup_datetime', and 'target' columns back to the train set features
tabular_data_train: pd.DataFrame = (
    Xtrain.assign(
        location_id=tabular_data.loc[Xtrain.index, "location_id"],
        pickup_datetime=tabular_data.loc[Xtrain.index, "pickup_datetime"], 
        target=ytrain
    )
    [tabular_data.columns]
)

# add the 'location_id', 'pickup_datetime', and 'target' columns back to the test set features
tabular_data_test: pd.DataFrame = (
    Xtest.assign(
        location_id=tabular_data.loc[Xtest.index, "location_id"],
        pickup_datetime=tabular_data.loc[Xtest.index, "pickup_datetime"], 
        target=ytest
    )
    [tabular_data.columns]
)

# vertically concatenate the 'tabular_data_train' and 'tabular_data_test' pd.DataFrames
tabular_data_confirm: pd.DataFrame = (
    pd.concat((tabular_data_train, tabular_data_test), axis=0).sort_index()
)

# confirm that the 'tabular_data' and 'tabular_data_confirm' pd.DataFrames are identical
assert np.mean(tabular_data.values == tabular_data_confirm.values) == 1

#### **`Inference pipeline and Streamlit web app`**

In [ ]:
# (1) fetch the latest hourly NYC taxi demand data from the Hopsworks project's Feature Group
# (2) transform the hourly NYC taxi demand data into ML-ready features and labels
# (3) load the model from the Hopsworks project's Model Registry
# (4) use the model to generate the one-step forecast, i.e., the taxi demand for the next hour

In [6]:
from importlib import reload

import src.config
import src.feature_store_api
import src.ingest
import src.inference
import src.pipelines.training_pipeline
import src.train
import src.transform

reload(src.config)
reload(src.feature_store_api)
reload(src.ingest)
reload(src.inference)
reload(src.pipelines.training_pipeline)
reload(src.train)
reload(src.transform)

from src.feature_store_api import HOPSWORKS_CONFIG
from src.ingest import load_taxi_zones
from src.inference import generate_forecast
from src.pipelines.training_pipeline import evaluate_model
from src.transform import fetch_and_transform, plot_record

In [2]:
# specify the starting and ending timestamps for which to ingest data
current_ts: pd.Timestamp = pd.Timestamp(datetime.now(timezone.utc)).floor("H")
end: pd.Timestamp = current_ts - pd.Timedelta(days=366)
start: pd.Timestamp = end - pd.Timedelta(days=28)

# a list of pre-processed pd.DataFrames, one per (year, month) pair
dfs: list[pd.DataFrame] = [
    download_file(year, month) 
    for year, month in zip([start.year, end.year], [start.month, end.month])
]

# (1) convert each pd.DataFrame in the 'dfs' list to a pl.DataFrame, and ...
# vertically concatenate them into a single pl.DataFrame
# (2) modify the 'pickup_datetime' column's timestamps by setting their timezone to UTC.
# Then, filter the pl.DataFrame by only keeping records (rows) whose modified timestamp is ...
# greater than or equal to the 'start' timestamp and less than or equal to the 'end' timestamp
# (3) modify the 'pickup_datetime' column's timestamps by ...
# (a) setting their timezone to UTC, ...
# (b) offsetting them forward in time by one year, and ...
# (c) converting them to integers that represent the time passed, in ms, since the Unix EPOCH
# (4) rename the 'pickup_datetime' column to 'unix_epoch_ms'
# (5) convert the pl.DataFrame to a pd.DataFrame
# NOTE: the NYC taxi data lags behind the current date by three months, so the timestamps are ...
# offset forward in time by one year to simulate a real time data ingestion scenario
# NOTE: the 'df' pd.DataFrame contains the data that will be uploaded to Hopsworks
df: pd.DataFrame = (
    pl.concat([pl.from_pandas(df) for df in dfs], how="vertical") # (1)
    .filter( # (2)
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").ge(start),
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").le(end)
    )
    .with_columns( # (3)
        pl.col("pickup_datetime")
        .dt.replace_time_zone("UTC")
        .dt.offset_by("1y")
        .dt.epoch(time_unit="ms")
    )
    .rename({"pickup_datetime": "unix_epoch_ms"}) # (4)
    .to_pandas() # (5)
)
df

2024-07-22 16:24:22,708 INFO: Downloading, validating, and pre-processing https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-06.parquet


100%|██████████| 258/258 [00:04<00:00, 59.22it/s]


2024-07-22 16:24:36,337 INFO: Downloading, validating, and pre-processing https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-07.parquet


100%|██████████| 255/255 [00:03<00:00, 64.83it/s]


,location_id,unix_epoch_ms,rides
0,1,1719270000000,1.0
1,1,1719273600000,0.0
2,1,1719277200000,0.0
3,1,1719280800000,0.0
4,1,1719284400000,0.0
...,...,...,...
161829,265,1721674800000,7.0
161830,265,1721678400000,3.0
161831,265,1721682000000,3.0
161832,265,1721685600000,3.0


In [3]:
# connect to the Hopsworks 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project="taxi_demand_forecasting",
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# connect to the project's feature store
feature_store: FeatureStore = project.get_feature_store()

# create the feature store's feature group
# NOTE: the feature group allows the 'df' pd.DataFrame's features to be saved to the feature store
# NOTE: the 'primary_key' parameter below, which can be set equal to a single column or ...
# a list of columns from the 'df' pd.DataFrame, is used as a unique identifier for each record (row)
feature_group: FeatureGroup = feature_store.get_or_create_feature_group(
    name="univariate_time_series",
    version=1,
    description="NYC taxi rides recorded at an hourly frequency",
    primary_key=["location_id", "unix_epoch_ms"],
    event_time="unix_epoch_ms",
    online_enabled=True
)

# # upload the 'df' pd.DataFrame to the 'taxi_demand_forecasting' project as a new feature group
# # NOTE: the feature group's name is, 'univariate_time_series', and it can be found on the ...
# # Hopsworks UI, under the 'taxi_demand_forecasting' project's 'Feature Group' tab 
feature_group.insert(df, write_options={"wait_for_job": False})

Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.
Feature Group created successfully, explore it at 
https://c.app.hopsworks.ai:443/p/708756/fs/704579/fg/1026424


Uploading Dataframe: 0.00% |          | Rows 0/161834 | Elapsed Time: 00:00 | Remaining Time: ?

Launching job: univariate_time_series_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai/p/708756/jobs/named/univariate_time_series_1_offline_fg_materialization/executions


(<hsfs.core.job.Job at 0x32ab624d0>, None)

In [7]:
HOPSWORKS_CONFIG

{'project': 'taxi_demand_forecasting',
 'feature_group': {'name': 'univariate_time_series',
  'version': 1,
  'description': 'NYC taxi rides recorded at an hourly frequency',
  'primary_key': ['location_id', 'unix_epoch_ms'],
  'event_time': 'unix_epoch_ms',
  'online_enabled': True},
 'model_registry': {'model_name': 'one_step_forecaster'}}

In [18]:
# output an ML-ready dataset of features and labels from the latest NYC taxi demand data
fetch_and_transform()

Connection closed.
Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.50s) 
2024-07-22 17:13:12,523 INFO: Transforming the hourly taxi demand data into ML-ready features and labels.


100%|██████████| 254/254 [00:08<00:00, 29.39it/s]


,location_id,pickup_datetime,hour,day_of_week,avg_24_lags,avg_20_lags,avg_16_lags,avg_12_lags,avg_8_lags,avg_4_lags,...,lag_9,lag_8,lag_7,lag_6,lag_5,lag_4,lag_3,lag_2,lag_1,target
0,1,2024-06-26 00:00:00,0,2,0.916667,1.10,1.0000,1.250000,0.750,0.50,...,5.0,0.0,0.0,3.0,1.0,1.0,0.0,1.0,0.0,0.0
1,1,2024-06-26 01:00:00,1,2,0.916667,1.10,1.0000,1.000000,0.750,0.25,...,0.0,0.0,3.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
2,1,2024-06-26 02:00:00,2,2,0.958333,1.10,1.0625,1.083333,0.875,0.50,...,0.0,3.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1,2024-06-26 03:00:00,3,2,0.958333,1.05,1.0000,1.000000,0.500,0.25,...,3.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,1,2024-06-26 04:00:00,4,2,0.958333,0.85,1.0000,0.583333,0.375,0.25,...,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155757,265,2024-07-22 20:00:00,20,0,4.416667,4.35,4.0000,5.083333,5.625,7.75,...,7.0,1.0,5.0,3.0,5.0,12.0,9.0,3.0,7.0,3.0
155758,265,2024-07-22 21:00:00,21,0,4.291667,4.10,4.0625,5.250000,5.875,5.50,...,1.0,5.0,3.0,5.0,12.0,9.0,3.0,7.0,3.0,3.0
155759,265,2024-07-22 22:00:00,22,0,4.291667,4.00,4.1875,5.333333,5.625,4.00,...,5.0,3.0,5.0,12.0,9.0,3.0,7.0,3.0,3.0,3.0
155760,265,2024-07-22 23:00:00,23,0,4.291667,3.85,4.3750,5.083333,5.625,4.00,...,3.0,5.0,12.0,9.0,3.0,7.0,3.0,3.0,3.0,3.0


In [19]:
# output the 10 busiest location IDs, based on their forecasted taxi demand
df_forecast: pd.DataFrame = (
    fetch_and_transform()
    .pipe(generate_forecast)
    .sort_values("forecast", ascending=False)
    .reset_index(drop=True)
    .head(10)
)
pl.from_pandas(df_forecast)

Connection closed.
Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.43s) 
2024-07-22 17:13:38,661 INFO: Transforming the hourly taxi demand data into ML-ready features and labels.


100%|██████████| 254/254 [00:08<00:00, 29.16it/s]

Connection closed.
Connected. Call `.close()` to terminate connection gracefully.



Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.


100%|██████████| 253/253 [00:00<00:00, 424.16it/s]


location_id,pickup_datetime,hour,day_of_week,avg_24_lags,avg_20_lags,avg_16_lags,avg_12_lags,avg_8_lags,avg_4_lags,lag_24,lag_23,lag_22,lag_21,lag_20,lag_19,lag_18,lag_17,lag_16,lag_15,lag_14,lag_13,lag_12,lag_11,lag_10,lag_9,lag_8,lag_7,lag_6,lag_5,lag_4,lag_3,lag_2,lag_1,forecast
i32,datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
79,2024-07-23 01:00:00,1.0,1.0,162.791667,129.8,155.4375,184.166667,222.25,294.0,489.0,440.0,270.0,112.0,31.0,24.0,23.0,31.0,45.0,77.0,73.0,82.0,118.0,90.0,115.0,109.0,122.0,150.0,145.0,185.0,186.0,240.0,343.0,407.0,410.0
249,2024-07-23 01:00:00,1.0,1.0,133.541667,117.9,143.0625,171.916667,209.875,287.5,370.0,274.0,152.0,51.0,10.0,13.0,10.0,36.0,44.0,51.0,62.0,69.0,78.0,81.0,110.0,115.0,104.0,138.0,148.0,139.0,213.0,275.0,297.0,365.0,381.0
148,2024-07-23 01:00:00,1.0,1.0,87.5,55.7,67.0,81.333333,103.0,153.75,369.0,306.0,217.0,94.0,13.0,11.0,7.0,11.0,21.0,21.0,27.0,27.0,30.0,46.0,38.0,38.0,36.0,44.0,66.0,63.0,66.0,100.0,172.0,277.0,332.0
114,2024-07-23 01:00:00,1.0,1.0,89.125,75.05,91.9375,110.583333,130.125,178.0,244.0,232.0,128.0,34.0,8.0,5.0,5.0,12.0,18.0,25.0,44.0,57.0,59.0,57.0,80.0,90.0,72.0,87.0,79.0,91.0,118.0,148.0,196.0,250.0,279.0
48,2024-07-23 01:00:00,1.0,1.0,127.291667,128.8,150.0625,164.333333,181.875,224.25,192.0,134.0,77.0,76.0,25.0,28.0,48.0,74.0,86.0,86.0,125.0,132.0,140.0,110.0,125.0,142.0,143.0,152.0,132.0,131.0,167.0,239.0,269.0,222.0,204.0
158,2024-07-23 01:00:00,1.0,1.0,59.333333,51.55,63.9375,77.833333,91.625,105.5,151.0,142.0,76.0,24.0,1.0,1.0,3.0,3.0,11.0,15.0,25.0,38.0,40.0,45.0,57.0,59.0,83.0,83.0,71.0,74.0,90.0,87.0,102.0,143.0,164.0
107,2024-07-23 01:00:00,1.0,1.0,101.75,106.8,122.9375,127.583333,133.5,144.0,122.0,110.0,56.0,18.0,20.0,22.0,52.0,75.0,91.0,110.0,122.0,113.0,122.0,125.0,112.0,104.0,92.0,107.0,142.0,151.0,148.0,131.0,146.0,151.0,135.0
234,2024-07-23 01:00:00,1.0,1.0,119.291667,128.7,157.0625,171.083333,173.25,165.5,142.0,85.0,46.0,16.0,5.0,8.0,19.0,29.0,60.0,99.0,137.0,164.0,164.0,170.0,174.0,159.0,163.0,177.0,192.0,192.0,159.0,154.0,192.0,157.0,130.0
144,2024-07-23 01:00:00,1.0,1.0,60.083333,55.1,68.25,84.166667,93.875,96.25,111.0,113.0,70.0,46.0,2.0,2.0,2.0,4.0,3.0,13.0,31.0,35.0,65.0,48.0,77.0,69.0,92.0,83.0,78.0,113.0,71.0,93.0,107.0,114.0,116.0


In [20]:
# display a line plot of one of the 10 busiest location IDs
fig: Figure = plot_record(df_forecast, np.random.choice(df_forecast["location_id"]))
fig.show()

In [21]:
# QC the current forecasting model
evaluate_model()

Connection closed.
Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (2.41s) 
2024-07-22 17:14:26,807 INFO: Transforming the hourly taxi demand data into ML-ready features and labels.


100%|██████████| 254/254 [00:08<00:00, 29.93it/s]


Connection closed.
Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.


100%|██████████| 253/253 [00:00<00:00, 430.63it/s]


2024-07-22 17:14:39,004 INFO: The current forecasting model is fine.


In [23]:
# output the gpd.GeoDataFrame containing the NYC taxi zones
load_taxi_zones()

,location_id,zone,geometry
0,1,Newark Airport,"POLYGON ((-74.18445 40.695, -74.18449 40.6951,..."
1,2,Jamaica Bay,"MULTIPOLYGON (((-73.82338 40.63899, -73.82277 ..."
2,3,Allerton/Pelham Gardens,"POLYGON ((-73.84793 40.87134, -73.84725 40.870..."
3,4,Alphabet City,"POLYGON ((-73.97177 40.72582, -73.97179 40.725..."
4,5,Arden Heights,"POLYGON ((-74.17422 40.56257, -74.17349 40.562..."
...,...,...,...
258,259,Woodlawn/Wakefield,"POLYGON ((-73.85107 40.91037, -73.85207 40.909..."
259,260,Woodside,"POLYGON ((-73.90175 40.76078, -73.90147 40.759..."
260,261,World Trade Center,"POLYGON ((-74.01333 40.70503, -74.01327 40.704..."
261,262,Yorkville East,"MULTIPOLYGON (((-73.94383 40.78286, -73.94376 ..."
